# Public portfolio notebook
Outputs have been removed before publication. Execution requires authorised local course data. Saved aggregate results are in `reports/portfolio/`; the public dashboard uses synthetic data.


# BNPL validation-only baseline models

## tl;dr

This notebook displays an existing checkpoint comparing two trailing-revenue rules, Ridge, and histogram gradient boosting on three
validation cutoffs. Learned models have all-feature and no-fraud variants. Test features and labels
remain sealed, and no merchant-level rows are displayed. It never starts model training.
The user has requested a complete training plan before any further fitting or test scoring.

## Context & Methods

All preprocessing is fitted on train. Model settings and the selection rule are fixed in
`config/baseline_v1.json`. NDCG@100 is the primary metric and oracle revenue capture is the
tiebreaker.

### Key Assumptions

Fraud fields represent evidence coverage rather than realised fraud. Three validation dates provide
limited evidence, and a selected development model is not a test-performance claim.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Image, Markdown

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/analysis_v1.json').exists())
sys.path.insert(0, str(root))
from src.bnpl_analysis.common import load_config
from src.run_artifacts import snapshot_manifest_path
config = load_config(root)
pd.set_option('display.max_columns', 12)


## Data
Only development split counts and aggregate results are shown.

In [ ]:
from src.bnpl_analysis.baselines import load_baseline_config
baseline_config = load_baseline_config(root)
metrics_path = root / 'reports/baseline' / baseline_config['baseline_run_id'] / 'metrics.json'
if not metrics_path.exists():
    raise FileNotFoundError('Saved baseline evidence missing; review the training plan before scheduling a new fit')
result = json.loads(metrics_path.read_text(encoding='utf-8'))
display(pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [result['development_protocol']['train_rows'], result['development_protocol']['validation_rows'], 'SEALED'],
    'role': ['fit preprocessing and models', 'compare fixed candidates', 'not opened'],
}))


## Results
Metrics are macro-averaged over the three validation cutoffs.

In [ ]:
metrics = pd.DataFrame(result['aggregate_validation_metrics'])
display(metrics)
print('Selected on validation:', result['selection']['selected_model'])
assert result['development_protocol']['preprocessing_fit_split'] == 'train'
assert result['development_protocol']['test_features_opened'] is False
assert result['development_protocol']['test_labels_opened'] is False
assert result['development_protocol']['test_evaluated'] is False
print('PASS: train-only preprocessing and sealed-test controls')


## Takeaways

The winning candidate is frozen as `validation_selected_not_tested` in
the versioned baseline report's `model_contract.json`. This notebook presents saved evidence;
explicit CLI runs produce the immutable model artifacts. Independent review is the next gate. Boruta and TCN are
later experiments; they must use the same development boundary and outperform these baselines before
the primary test is opened once.
